In [ ]:
%sql
/* ===================================================================================
   VIEW: vw_cost_center_mapping_bootstrap
   SOURCE: hive_metastore.ra_adido_2025.fy25_cost_center_mapping
   
   BUSINESS RULES & LOGIC APPLIED:
   
   1. MUTUALLY EXCLUSIVE BRANCHING (The "Yes" Rule) - **UPDATED for Case Insensitivity**:
      - If Col E is 'YES' (or blank): Use ONLY Primary Columns B, C, D. Ignore E & F.
      - If Col E is NOT 'YES': Use ONLY Columns E & F. Ignore B, C, D entirely.
      
   2. DIRTY STRING PARSING (Mashed Concatenations):
      - Col E and F often contain multiple Assessable Units mashed together without 
        spacing (e.g., "123456 - Name - Seg123457 - Name - Seg").
      - Uses Regex to slice the string into individual blocks strictly at every 6-digit boundary.
      
   3. HYPHEN PROTECTION & FALLBACK:
      - Extracts the 6-digit AU_ID.
      - For the remaining text: If hyphens exist, splits Name and Segment based on the 
        LAST hyphen (protecting internal hyphens like "Retail - Banking").
      - If no hyphens exist, treats the entire text as the AU Name with a blank Segment.
      
   4. DATA CLEANSING & ZERO-PADDING:
      - Strips the hidden ".0" ghost appended by Databricks/Excel imports.
      - ZERO-PADDING: Forces purely numeric Cost Center IDs under 4 digits to be 
        exactly 4 digits (e.g., '419' -> '0419') without truncating 5+ digit IDs.
      - Strips rogue leading/trailing double quotes (caused by CSV escaping).
      - Converts double/multiple spaces into a single space.
      - Strictly drops any ghost rows where the AU_ID or AU_Name is missing/blank.
      
   5. SINGLE SOURCE OF TRUTH (Deduplication & Standardization):
      - Uses UNION and DISTINCT to remove duplicate AU mappings per Cost Center.
      - Resolves naming conflicts across different Cost Centers (e.g., "Insurance" vs 
        "General Insurance" for the same ID) by using a Window Function to force a 
        single, standardized AU Name and Segment per AU_ID alphabetically.
=================================================================================== */

CREATE OR REPLACE TEMP VIEW vw_cost_center_mapping_bootstrap AS

WITH Raw_Import AS (
    -- Step 0a: Pull raw columns and fix the "Excel .0 Ghost" issue
    SELECT 
        -- Strips trailing '.0' so '419.0' correctly becomes '419' before we pad it
        REGEXP_REPLACE(TRIM(CAST(`CostCenterId` AS STRING)), '\\.0$', '') AS Raw_CC_String,
        TRIM(CAST(`AssessableUnitID` AS STRING)) AS Primary_AU_ID,
        TRIM(`AssessableUnitName`) AS Primary_AU_Name,
        TRIM(`Segment`) AS Primary_Segment,
        TRIM(`AdditionalAssessableUnitIDandNameandSegment`) AS Col_E,
        TRIM(`AdditionalAUID`) AS Col_F
    FROM hive_metastore.ra_adido_2025.fy25_cost_center_mapping
),

Base_Data AS (
    -- Step 0b: Apply the safe 4-digit ZERO-PADDING rule
    SELECT 
        CASE 
            WHEN Raw_CC_String RLIKE '^[0-9]+$' AND LENGTH(Raw_CC_String) < 4 
            THEN LPAD(Raw_CC_String, 4, '0')
            ELSE Raw_CC_String
        END AS Cost_Center_ID,
        Primary_AU_ID,
        Primary_AU_Name,
        Primary_Segment,
        Col_E,
        Col_F
    FROM Raw_Import
),

Branch_Primary AS (
    -- LOGIC 1a: FIXED - Made Case-Insensitive to protect "YES" strings
    SELECT 
        Cost_Center_ID,
        Primary_AU_ID AS AU_ID,
        Primary_AU_Name AS AU_Name,
        Primary_Segment AS Segment_Name
    FROM Base_Data
    WHERE UPPER(TRIM(Col_E)) = 'YES' 
       OR Col_E IS NULL 
       OR Col_E = ''
),

Branch_Additional_Raw AS (
    -- LOGIC 1b: FIXED - Made Case-Insensitive
    SELECT 
        Cost_Center_ID,
        CONCAT_WS(' ', COALESCE(Col_E, ''), COALESCE(Col_F, '')) AS Mashed_String
    FROM Base_Data
    WHERE UPPER(TRIM(Col_E)) != 'YES' 
      AND Col_E IS NOT NULL 
      AND Col_E != ''
),

Extracted_Blocks AS (
    -- LOGIC 2: Slice the mashed E & F text into blocks at every 6-digit boundary
    SELECT 
        Cost_Center_ID,
        EXPLODE(regexp_extract_all(Mashed_String, '([0-9]{6}.*?(?=[0-9]{6}|$))')) AS Raw_Block
    FROM Branch_Additional_Raw
    WHERE Mashed_String != ''
),

Separated_ID_And_Remainder AS (
    -- LOGIC 3a: Pull out the 6-digit ID, and isolate the rest of the text
    SELECT 
        Cost_Center_ID,
        TRIM(regexp_extract(Raw_Block, '^([0-9]{6})', 1)) AS AU_ID,
        TRIM(REGEXP_REPLACE(Raw_Block, '^[0-9]{6}[ \t-]*', '')) AS Remainder
    FROM Extracted_Blocks
    WHERE TRIM(Raw_Block) != ''
),

Parsed_Additionals AS (
    -- LOGIC 3b: Smartly parse the remainder based on whether hyphens exist
    SELECT 
        Cost_Center_ID,
        AU_ID,
        CASE 
            WHEN Remainder LIKE '%-%' THEN TRIM(regexp_extract(Remainder, '^(.*)[ \t]*-[ \t]*[^-]+$', 1))
            ELSE Remainder 
        END AS AU_Name,
        CASE 
            WHEN Remainder LIKE '%-%' THEN TRIM(regexp_extract(Remainder, '.*[ \t]*-[ \t]*([^-]+)$', 1))
            ELSE '' 
        END AS Segment_Name
    FROM Separated_ID_And_Remainder
),

Cleaned_Stack AS (
    -- LOGIC 4: Combine BOTH branches, clean quotes, remove weird spaces, drop blanks
    SELECT DISTINCT 
        Cost_Center_ID, 
        AU_ID, 
        TRIM(REGEXP_REPLACE(REGEXP_REPLACE(AU_Name, '^"|"$', ''), '[ ]+', ' ')) AS AU_Name, 
        TRIM(REGEXP_REPLACE(REGEXP_REPLACE(Segment_Name, '^"|"$', ''), '[ ]+', ' ')) AS Segment_Name 
    FROM (
        SELECT Cost_Center_ID, AU_ID, AU_Name, Segment_Name FROM Branch_Primary
        UNION
        SELECT Cost_Center_ID, AU_ID, AU_Name, Segment_Name FROM Parsed_Additionals
    )
    WHERE AU_ID IS NOT NULL AND AU_ID != ''
      AND AU_Name IS NOT NULL AND TRIM(AU_Name) != ''
)

-- LOGIC 5: Final Standardization (One Name/Segment per AU_ID)
SELECT DISTINCT 
    Cost_Center_ID,
    AU_ID,
    FIRST_VALUE(AU_Name) OVER (PARTITION BY AU_ID ORDER BY AU_Name ASC) AS AU_Name,
    FIRST_VALUE(Segment_Name) OVER (PARTITION BY AU_ID ORDER BY AU_Name ASC) AS Segment_Name
FROM Cleaned_Stack;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: vw_cost_center_mapping_bootstrap_DEBUG
   
   PURPOSE: A universal traceability query that tracks every Cost Center through the 
   parsing pipeline, explicitly printing routing decisions, intermediate string cuts, 
   and the final keep/drop status.
   
   KEY UPGRADE: Uses `explode_outer` so that if a row fails the Regex extraction, 
   it doesn't vanish—it stays in the report so we can clearly label it as a failure!
=================================================================================== */

WITH Raw_Import AS (
    SELECT 
        REGEXP_REPLACE(TRIM(CAST(`CostCenterId` AS STRING)), '\\.0$', '') AS Raw_CC_String,
        TRIM(CAST(`AssessableUnitID` AS STRING)) AS Raw_Primary_AU_ID,
        TRIM(`AssessableUnitName`) AS Raw_Primary_AU_Name,
        TRIM(`Segment`) AS Raw_Primary_Segment,
        TRIM(`AdditionalAssessableUnitIDandNameandSegment`) AS Raw_Col_E,
        TRIM(`AdditionalAUID`) AS Raw_Col_F
    FROM hive_metastore.ra_adido_2025.fy25_cost_center_mapping
),

Base_Data AS (
    SELECT 
        CASE 
            WHEN Raw_CC_String RLIKE '^[0-9]+$' AND LENGTH(Raw_CC_String) < 4 THEN LPAD(Raw_CC_String, 4, '0')
            ELSE Raw_CC_String
        END AS Padded_CC_ID,
        Raw_Primary_AU_ID, Raw_Primary_AU_Name, Raw_Primary_Segment, Raw_Col_E, Raw_Col_F,
        
        -- TRACKER 1: Print the routing decision based on the case-insensitive 'Yes' rule
        CASE 
            WHEN UPPER(TRIM(Raw_Col_E)) = 'YES' OR Raw_Col_E IS NULL OR Raw_Col_E = '' THEN 'PRIMARY BRANCH (Cols B,C,D)'
            ELSE 'ADDITIONAL BRANCH (Regex on Cols E,F)'
        END AS Routing_Decision
        
    FROM Raw_Import
),

-- ==========================================
-- SIMULATE BRANCH 1 (PRIMARY)
-- ==========================================
Branch_Primary_Sim AS (
    SELECT 
        Padded_CC_ID, Raw_Primary_AU_ID, Raw_Primary_AU_Name, Raw_Primary_Segment, Raw_Col_E, Raw_Col_F, Routing_Decision,
        Raw_Primary_AU_ID AS Extracted_AU_ID,
        Raw_Primary_AU_Name AS Extracted_AU_Name,
        Raw_Primary_Segment AS Extracted_Segment,
        'N/A - Direct Pull' AS Regex_Mashed_String,
        'N/A - Direct Pull' AS Regex_Raw_Block
    FROM Base_Data
    WHERE Routing_Decision = 'PRIMARY BRANCH (Cols B,C,D)'
),

-- ==========================================
-- SIMULATE BRANCH 2 (ADDITIONAL REGEX)
-- ==========================================
Branch_Additional_Sim AS (
    SELECT 
        Padded_CC_ID, Raw_Primary_AU_ID, Raw_Primary_AU_Name, Raw_Primary_Segment, Raw_Col_E, Raw_Col_F, Routing_Decision,
        CONCAT_WS(' ', COALESCE(Raw_Col_E, ''), COALESCE(Raw_Col_F, '')) AS Regex_Mashed_String
    FROM Base_Data
    WHERE Routing_Decision = 'ADDITIONAL BRANCH (Regex on Cols E,F)'
),

Branch_Additional_Exploded AS (
    SELECT 
        b.*,
        -- EXPLODE_OUTER keeps the row alive even if the regex finds 0 matches!
        exploded_block AS Regex_Raw_Block
    FROM Branch_Additional_Sim b
    LATERAL VIEW explode_outer(regexp_extract_all(Regex_Mashed_String, '([0-9]{6}.*?(?=[0-9]{6}|$))')) AS exploded_block
),

Branch_Additional_Parsed AS (
    SELECT 
        Padded_CC_ID, Raw_Primary_AU_ID, Raw_Primary_AU_Name, Raw_Primary_Segment, Raw_Col_E, Raw_Col_F, Routing_Decision,
        Regex_Mashed_String,
        Regex_Raw_Block,
        
        -- Parse ID
        TRIM(regexp_extract(Regex_Raw_Block, '^([0-9]{6})', 1)) AS Extracted_AU_ID,
        
        -- Parse Name (Splitting on last hyphen if it exists)
        CASE 
            WHEN TRIM(REGEXP_REPLACE(Regex_Raw_Block, '^[0-9]{6}[ \t-]*', '')) LIKE '%-%' 
            THEN TRIM(regexp_extract(TRIM(REGEXP_REPLACE(Regex_Raw_Block, '^[0-9]{6}[ \t-]*', '')), '^(.*)[ \t]*-[ \t]*[^-]+$', 1))
            ELSE TRIM(REGEXP_REPLACE(Regex_Raw_Block, '^[0-9]{6}[ \t-]*', '')) 
        END AS Extracted_AU_Name,
        
        -- Parse Segment
        CASE 
            WHEN TRIM(REGEXP_REPLACE(Regex_Raw_Block, '^[0-9]{6}[ \t-]*', '')) LIKE '%-%' 
            THEN TRIM(regexp_extract(TRIM(REGEXP_REPLACE(Regex_Raw_Block, '^[0-9]{6}[ \t-]*', '')), '.*[ \t]*-[ \t]*([^-]+)$', 1))
            ELSE '' 
        END AS Extracted_Segment
        
    FROM Branch_Additional_Exploded
),

-- ==========================================
-- COMBINE & EVALUATE FINAL FILTERS
-- ==========================================
Combined_Sim AS (
    SELECT * FROM Branch_Primary_Sim
    UNION ALL
    SELECT * FROM Branch_Additional_Parsed
)

SELECT 
    Padded_CC_ID AS Final_CC_ID,
    Routing_Decision,
    
    -- Inputs for Traceability
    Raw_Col_E,
    Regex_Mashed_String AS String_Fed_To_Regex,
    Regex_Raw_Block AS Chunk_Extracted_By_Regex,
    
    -- Outputs after parsing/cleaning
    Extracted_AU_ID AS Final_Parsed_AU_ID,
    TRIM(REGEXP_REPLACE(REGEXP_REPLACE(Extracted_AU_Name, '^"|"$', ''), '[ ]+', ' ')) AS Final_Parsed_AU_Name,
    TRIM(REGEXP_REPLACE(REGEXP_REPLACE(Extracted_Segment, '^"|"$', ''), '[ ]+', ' ')) AS Final_Parsed_Segment,
    
    -- TRACKER 2: The Ultimate Keep/Drop Status
    CASE 
        WHEN Routing_Decision LIKE 'ADDITIONAL%' AND (Regex_Raw_Block IS NULL OR Regex_Raw_Block = '') 
            THEN '❌ DROPPED: Sent to Regex, but string contained no valid 6-digit ID.'
        WHEN Extracted_AU_ID IS NULL OR Extracted_AU_ID = '' 
            THEN '❌ DROPPED: Required AU_ID is blank.'
        WHEN Extracted_AU_Name IS NULL OR TRIM(Extracted_AU_Name) = '' 
            THEN '❌ DROPPED: Required AU_Name is blank.'
        ELSE '✅ KEPT: Successfully passed all parsing and filters.'
    END AS Pipeline_Status

FROM Combined_Sim
-- Uncomment the line below to test a specific broken Cost Center!
-- WHERE Padded_CC_ID = '2830'
ORDER BY Final_CC_ID;